# Aula 10 — Atenção

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

O PyTorch já vem instalado no Google Colab. Nenhuma instalação é
necessária, e nada aqui precisa de placa de vídeo.

## Parte A: Demonstração

### O embutimento: de número inteiro para vetor

O token da Aula 9 é um inteiro, e inteiro não serve para conta de
parecença. O modelo troca cada token por uma linha de uma tabela.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn

torch.manual_seed(42)

TAMANHO_VOCABULARIO = 1024
DIMENSAO = 128

embutir = nn.Embedding(TAMANHO_VOCABULARIO, DIMENSAO)

tokens = torch.tensor([[79, 600, 281, 111, 354]])   # "O menino era pai"
vetores = embutir(tokens)

print(f"entraram {tokens.shape} tokens")
print(f"saíram   {vetores.shape} (lote, tokens, números por token)")
print(f"pesos da tabela: {embutir.weight.numel():,}")
print()
print("os 8 primeiros números do token 600:")
print(vetores[0, 1, :8])

### O produto escalar mede parecença

Multiplica casa com casa e soma. É a mesma conta do filtro da Aula 8.

$$q \cdot k = \sum_{i=1}^{d} q_i k_i$$

In [ ]:
q = torch.tensor([0.0, 2.0, 0.0, 0.0])

for nome, k in [("mesmo sentido", torch.tensor([0.0, 2.0, 0.0, 0.0])),
                ("metade", torch.tensor([0.0, 1.0, 0.0, 0.0])),
                ("perpendicular", torch.tensor([1.0, 0.0, 0.0, 0.0])),
                ("sentido contrário", torch.tensor([0.0, -2.0, 0.0, 0.0]))]:
    print(f"{nome:<20} q·k = {torch.dot(q, k):>5.1f}")

### Atenção sem peso nenhum, em quatro passos

Antes de qualquer coisa treinável: três palavras, quatro números por
palavra, e a palavra "dorme" perguntando para onde olhar. Nenhuma tabela
de pesos entra nesta célula.

In [ ]:
palavras = ["o", "gato", "dorme"]

# Os vetores das três palavras. É só isso que existe por enquanto.
X = torch.tensor([[2.0, 0.0, 0.0, 0.0],    # o
                  [0.0, 3.0, 1.0, 0.0],    # gato
                  [0.0, 1.0, 1.0, 0.0]])   # dorme

consulta = X[2]                            # "dorme" pergunta

produtos = X @ consulta                    # 1. parecença com cada palavra
escalados = produtos / 4 ** 0.5            # 2. segura o tamanho
pesos = F.softmax(escalados, dim=-1)       # 3. vira peso que soma 1
saida = pesos @ X                          # 4. soma as três, pesada

for indice, palavra in enumerate(palavras):
    print(f"{palavra:<7} produto {produtos[indice]:>4.0f} -> "
          f"dividido {escalados[indice]:>4.0f} -> peso {pesos[indice]:.0%}")
print()
print("saída de 'dorme':", saida)
print()
print("Ela saiu carregando dois terços de 'gato'. E não usamos peso nenhum.")

### Buraco 1: sem pesos, a palavra olha para si mesma

Faça a mesma conta para as três posições e olhe a matriz inteira.

In [ ]:
def atencao_sem_pesos(vetores):
    pontos = vetores @ vetores.T / vetores.shape[1] ** 0.5
    pesos = F.softmax(pontos, dim=-1)
    return pesos, pesos @ vetores


pesos_todos, saidas = atencao_sem_pesos(X)

print(f"{'':<9}" + "".join(f"{p:>9}" for p in palavras))
for i, palavra in enumerate(palavras):
    linha = "".join(f"{100 * pesos_todos[i, j]:>8.0f}%" for j in range(3))
    marca = " <- olha para si mesma" if int(pesos_todos[i].argmax()) == i else ""
    print(f"{palavra:<9}{linha}{marca}")

print()
print("O produto de um vetor com ele mesmo é o comprimento ao quadrado:")
print("o maior valor que ele tira de qualquer comparação. Daí o espelho.")

### Buraco 2: embaralhe a frase e nada muda

"o gato dorme" contra "dorme gato o". Compare a saída da palavra
"dorme" nos dois casos.

In [ ]:
_, saidas_trocadas = atencao_sem_pesos(X[[2, 1, 0]])

print("saída de 'dorme' em 'o gato dorme':", saidas[2])
print("saída de 'dorme' em 'dorme gato o':", saidas_trocadas[0])
print()
print("Iguais:", torch.allclose(saidas[2], saidas_trocadas[0]))
print()
print("A fórmula não sabe onde cada palavra está. Ela vê um saco de")
print("palavras. Quem tapa esse buraco é o RoPE, no fim da aula.")

### Agora com pesos: Q, K e V

As três tabelas resolvem o buraco 1. Como Q e K saem de tabelas
diferentes, o produto de uma palavra com ela mesma deixa de ganhar
sozinho, e o treino decide quem olha para quem.

In [ ]:
torch.manual_seed(3)
tabela_q = nn.Linear(4, 4, bias=False)
tabela_k = nn.Linear(4, 4, bias=False)
tabela_v = nn.Linear(4, 4, bias=False)

Q, K, V = tabela_q(X), tabela_k(X), tabela_v(X)
pesos_com_pesos = F.softmax(Q @ K.T / 4 ** 0.5, dim=-1)

print("sem pesos, quem cada palavra mais olha:",
      [palavras[i] for i in pesos_todos.argmax(dim=1)])
print("com pesos, quem cada palavra mais olha:",
      [palavras[i] for i in pesos_com_pesos.argmax(dim=1)])
print()
print("São tabelas sorteadas, ainda sem treino: o ponto é só que a")
print("diagonal deixou de vencer sozinha.")

### A atenção inteira, para todas as posições de uma vez

$$\text{Atenção}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d}}\right) V$$

Quatro linhas de código, e uma delas é a máscara.

In [ ]:
def atencao(Q, K, V):
    tokens = Q.shape[0]
    dimensao = Q.shape[1]
    pontos = Q @ K.T / dimensao ** 0.5
    mascara = torch.triu(torch.ones(tokens, tokens), diagonal=1).bool()
    pontos = pontos.masked_fill(mascara, float("-inf"))
    pesos = F.softmax(pontos, dim=-1)
    return pesos @ V, pesos


torch.manual_seed(7)
Q_todos = torch.randn(6, 4)
K_todos = torch.randn(6, 4)
V_todos = torch.randn(6, 4)

_, pesos_todos = atencao(Q_todos, K_todos, V_todos)

print("matriz de pesos (cada linha soma 1):")
print(pesos_todos.round(decimals=2))
print()
print("soma de cada linha:", pesos_todos.sum(dim=1))

### O que a máscara impede

Sem ela, a posição 3 olha a posição 4, que é justamente a palavra que ela
deveria prever.

In [ ]:
pontos = Q_todos @ K_todos.T / 4 ** 0.5
sem_mascara = F.softmax(pontos, dim=-1)

print("sem máscara, linha 1 (a primeira palavra):")
print(sem_mascara[0].round(decimals=2))
print("ela está olhando cinco palavras que ainda não existem.")
print()
print("com máscara, linha 1:")
print(pesos_todos[0].round(decimals=2))
print("100% em si mesma, que é tudo o que ela tem.")

### A atenção do modelo treinado

Agora a mesma conta, dentro do modelo das Aulas 11 e 12. O código do
modelo aparece inteiro na Aula 11: aqui só carregamos os pesos.

In [ ]:
import json
import re
import urllib.request

URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"
# Alternativa para testar offline:
# URL_DADOS = "../../data/"


def baixar(nome, binario=False):
    if URL_DADOS.startswith("http"):
        dados = urllib.request.urlopen(URL_DADOS + nome).read()
        return dados if binario else dados.decode("utf-8")
    modo = "rb" if binario else "r"
    with open(URL_DADOS + nome, modo, **({} if binario else {"encoding": "utf-8"})) as f:
        return f.read()


configuracao = json.loads(baixar("machado_bpe.json"))
PADRAO = re.compile(configuracao["padrao"])
fusoes = {(a, b): 256 + i for i, (a, b) in enumerate(configuracao["fusoes"])}

tabela = {i: bytes([i]) for i in range(256)}
for (a, b), novo in fusoes.items():
    tabela[novo] = tabela[a] + tabela[b]


def codificar(texto):
    saida = []
    for pedaco in PADRAO.findall(texto):
        simbolos = list(pedaco.encode("utf-8"))
        while len(simbolos) >= 2:
            candidatos = [p for p in zip(simbolos, simbolos[1:]) if p in fusoes]
            if not candidatos:
                break
            a, b = min(candidatos, key=lambda p: fusoes[p])
            novos, i = [], 0
            while i < len(simbolos):
                if i < len(simbolos) - 1 and (simbolos[i], simbolos[i + 1]) == (a, b):
                    novos.append(fusoes[(a, b)])
                    i += 2
                else:
                    novos.append(simbolos[i])
                    i += 1
            simbolos = novos
        saida.extend(simbolos)
    return saida


def decodificar(ids):
    return b"".join(tabela[int(i)] for i in ids).decode("utf-8", errors="replace")


print(codificar("A casa de Capitú"))
print([decodificar([n]) for n in codificar("A casa de Capitú")])

In [ ]:
import io

import matplotlib.pyplot as plt

CABECAS = 4
CAMADAS = 4

guardado = torch.load(io.BytesIO(baixar("mini_llm.pt", binario=True)),
                      weights_only=False)
pesos_do_modelo = guardado["pesos"]

print(f"modelo treinado até o passo {guardado['passo']}")
print(f"perda de validação: {guardado['perda_validacao']:.3f}")
print()
for nome in list(pesos_do_modelo)[:6]:
    print(f"  {nome:<28} {tuple(pesos_do_modelo[nome].shape)}")

In [ ]:
def girar(x, base=10000.0):
    """RoPE: gira o vetor por um ângulo proporcional à posição."""
    lote, cabecas, tokens, dim = x.shape
    metade = dim // 2
    frequencias = base ** (-torch.arange(0, metade).float() / metade)
    angulos = torch.arange(tokens).float()[:, None] * frequencias[None, :]
    cosseno, seno = angulos.cos()[None, None], angulos.sin()[None, None]
    primeira, segunda = x[..., :metade], x[..., metade:]
    return torch.cat([primeira * cosseno - segunda * seno,
                      primeira * seno + segunda * cosseno], dim=-1)


def pesos_de_atencao(ids, camada, usar_rope=True):
    """Refaz a conta da atenção da primeira camada, com os pesos treinados.

    Para as camadas de cima o resultado seria aproximado, porque falta
    rodar os blocos anteriores. Por isso o notebook fica na camada 1.
    """
    x = F.embedding(torch.tensor([ids]), pesos_do_modelo["embutir.weight"])
    peso_norma = pesos_do_modelo[f"blocos.{camada}.norma1.weight"]
    x = F.rms_norm(x, (DIMENSAO,), peso_norma)
    qkv = x @ pesos_do_modelo[f"blocos.{camada}.atencao.qkv.weight"].T
    pergunta, chave, _ = qkv.split(DIMENSAO, dim=2)
    forma = (1, len(ids), CABECAS, DIMENSAO // CABECAS)
    pergunta = pergunta.view(forma).transpose(1, 2)
    chave = chave.view(forma).transpose(1, 2)
    if usar_rope:
        pergunta, chave = girar(pergunta), girar(chave)
    pontos = pergunta @ chave.transpose(-2, -1) / (DIMENSAO // CABECAS) ** 0.5
    mascara = torch.triu(torch.ones(len(ids), len(ids)), diagonal=1).bool()
    pontos = pontos.masked_fill(mascara, float("-inf"))
    return F.softmax(pontos, dim=-1)[0]


frase = "A casa de Capitú tinha uma janela para a rua"
ids = codificar(frase)
rotulos = [decodificar([i]).replace(" ", "·") for i in ids]

with torch.no_grad():
    mapas = pesos_de_atencao(ids, camada=0)

plt.figure(figsize=(12, 5))
for cabeca in range(2):
    plt.subplot(1, 2, cabeca + 1)
    plt.imshow(mapas[cabeca], cmap="Reds")
    plt.xticks(range(len(rotulos)), rotulos, rotation=60, ha="right", fontsize=8)
    plt.yticks(range(len(rotulos)), rotulos, fontsize=8)
    plt.title(f"camada 1, cabeça {cabeca + 1}")
plt.tight_layout()
plt.show()

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: o tamanho do embutimento

Rode e observe: quantos pesos tem a tabela de embutimento, e quantos
números representam cada token?

In [ ]:
print(f"formato da tabela: {tuple(embutir.weight.shape)}")
print(f"pesos: {embutir.weight.numel():,}")
print(f"fração do modelo inteiro: {embutir.weight.numel() / 787_584:.0%}")

In [ ]:
if embutir.weight.numel() == 131_072:
    print("✅ 131.072 pesos, ou 1.024 tokens × 128 números.")
    print("   É um sexto do modelo inteiro, só para a porta de entrada.")
else:
    print("❌ Confira se a célula que cria o embutimento rodou.")

### Exercício 2: os produtos escalares

Calcule `meus_produtos`: o produto escalar de `minha_pergunta` com cada
uma das três linhas de `K`.

Dica: `K @ minha_pergunta` faz os três de uma vez.

In [ ]:
minha_pergunta = torch.tensor([1.0, 1.0, 0.0, 0.0])

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for indice, palavra in enumerate(palavras):
    print(f"{palavra:<7} {meus_produtos[indice]:>5.1f}")

In [ ]:
if torch.allclose(meus_produtos, torch.tensor([1.0, 2.0, 1.0])):
    print("✅ Produtos 1, 2 e 1.")
    print("   Esta pergunta olha um pouco para tudo, sem escolher nada.")
else:
    print(f"❌ Esperava [1, 2, 1] e veio {meus_produtos.tolist()}.")

### Exercício 3: da pontuação para o peso

Divida `meus_produtos` pela raiz da dimensão (que é 4) e aplique a
softmax. Guarde em `meus_pesos`.

$$\text{peso}_j = \text{softmax}\!\left(\frac{q \cdot k_j}{\sqrt{d}}\right)$$

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for indice, palavra in enumerate(palavras):
    print(f"{palavra:<7} {meus_pesos[indice]:.1%}")
print(f"soma: {meus_pesos.sum():.2f}")

In [ ]:
if abs(float(meus_pesos.sum()) - 1.0) < 0.001 and meus_pesos.argmax() == 1:
    print("✅ Os três somam 1, e 'gato' levou a maior fatia.")
else:
    print("❌ Confira: divida por 4**0.5 e depois use F.softmax(..., dim=-1).")

### Exercício 4: a máscara causal

Monte `minha_mascara`, uma matriz 5×5 de `True` acima da diagonal e
`False` no resto.

Dica: `torch.triu(torch.ones(5, 5), diagonal=1).bool()`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minha_mascara)

In [ ]:
if minha_mascara.sum() == 10 and not minha_mascara[4, 0]:
    print("✅ Dez posições bloqueadas: as que apontam para o futuro.")
    print("   A última linha não tem nenhuma, porque ela pode olhar tudo.")
else:
    print("❌ Confira o diagonal=1: com diagonal=0 você bloqueia a própria posição.")

### Exercício 5: a atenção inteira

Escreva `minha_atencao(Q, K, V)`, que devolve a saída e os pesos. Quatro
passos: produtos com escala, máscara, softmax, mistura de V.

In [ ]:
torch.manual_seed(3)
Qx, Kx, Vx = torch.randn(4, 8), torch.randn(4, 8), torch.randn(4, 8)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_saida, meus_mapas = minha_atencao(Qx, Kx, Vx)
print(meus_mapas.round(decimals=2))

In [ ]:
if (minha_saida.shape == (4, 8)
        and torch.allclose(meus_mapas.sum(dim=1), torch.ones(4))
        and float(meus_mapas[0, 1]) == 0.0):
    print("✅ Saída no formato certo, linhas somando 1, e nada acima da diagonal.")
else:
    print("❌ Confira a ordem: escala, máscara, softmax, e só então multiplicar por V.")

### Exercício 6: lendo o mapa do modelo treinado

Rode e observe: quanto peso cada cabeça dá à palavra imediatamente
anterior?

In [ ]:
with torch.no_grad():
    mapas_da_frase = pesos_de_atencao(ids, camada=0)

for cabeca in range(CABECAS):
    mapa = mapas_da_frase[cabeca]
    anterior = sum(float(mapa[i, i - 1]) for i in range(1, len(ids))) / (len(ids) - 1)
    propria = sum(float(mapa[i, i]) for i in range(1, len(ids))) / (len(ids) - 1)
    print(f"cabeça {cabeca + 1}: {anterior:.0%} na palavra anterior, "
          f"{propria:.0%} nela mesma")

In [ ]:
print("Converse com um colega: uma cabeça que só olha a palavra anterior")
print("serve para quê? Pense em concordância de gênero e número.")

### Exercício 7: desafio, a atenção sem RoPE

Calcule os mapas da mesma frase **sem** o giro da posição
(`usar_rope=False`) e guarde em `sem_posicao`. Depois compare com
`mapas_da_frase`.

A diferença mostra o quanto a posição pesa na decisão do modelo.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
diferenca = (sem_posicao - mapas_da_frase).abs().mean(dim=(1, 2))
for cabeca in range(CABECAS):
    print(f"cabeça {cabeca + 1}: mudança média de {diferenca[cabeca]:.3f}")

plt.figure(figsize=(11, 4.5))
for coluna, (mapa, titulo) in enumerate(
        [(mapas_da_frase[0], "com RoPE"), (sem_posicao[0], "sem RoPE")]):
    plt.subplot(1, 2, coluna + 1)
    plt.imshow(mapa, cmap="Reds")
    plt.xticks(range(len(rotulos)), rotulos, rotation=60, ha="right", fontsize=8)
    plt.yticks(range(len(rotulos)), rotulos, fontsize=8)
    plt.title(f"cabeça 1, {titulo}")
plt.tight_layout()
plt.show()

In [ ]:
if float(diferenca.max()) > 0.005:
    print("✅ Tirar o giro muda os mapas: o modelo aprendeu a contar com a posição.")
else:
    print("❌ Se não mudou nada, confira se você passou usar_rope=False.")

Agora, em texto: explique com suas palavras por que a máscara causal é
obrigatória, e o que aconteceria com um modelo treinado sem ela. Edite
esta célula (duplo clique nela) e escreva sua resposta no lugar deste
parágrafo.